# 00 — Context v2: Frequency Multiplexing Enables Quantum Scale

**Version marker:** `00_context_v2_architecture_notebook`

**Seminar:** Integrated Microcombs for Quantum Applications  
**Speaker:** Xu Yi, University of Virginia

This notebook frames the repository question:

> **Which resource scales quantum systems: more devices or more modes?**

Notebook 00 is architectural. It intentionally does **not** model full quantum optics, fabrication, detection, loss, or Kerr dynamics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

VERSION = "00_context_v2_architecture_notebook"
print("running:", VERSION)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd

FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csv"
JSON_DIR = RESULTS_DIR / "json"

for path in [FIGURES_DIR, CSV_DIR, JSON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. Architecture table

The question is not whether both approaches can label \(N\) channels.

The question is which resource has to scale:

- replicated devices
- or frequency-indexed modes

In [ ]:
architecture_table = pd.DataFrame([
    {"resource": "Channels", "scale_by_devices": "N", "scale_by_modes": "N", "interpretation": "same target channel count"},
    {"resource": "Sources", "scale_by_devices": "N independent sources", "scale_by_modes": "1 pump source", "interpretation": "mode scaling reuses one source path"},
    {"resource": "Resonators / devices", "scale_by_devices": "N devices", "scale_by_modes": "1 integrated microresonator", "interpretation": "mode scaling increases internal mode count rather than device count"},
    {"resource": "Frequency modes", "scale_by_devices": "not the primary scaling resource", "scale_by_modes": "N frequency-indexed modes", "interpretation": "frequency becomes the multiplexing dimension"},
    {"resource": "Optical paths", "scale_by_devices": "N optical paths", "scale_by_modes": "1 to few shared paths", "interpretation": "complexity shifts from replicated paths to mode addressing"},
])
architecture_table

In [ ]:
architecture_table_path = CSV_DIR / "00_v2_architecture_table.csv"
architecture_table.to_csv(architecture_table_path, index=False)
print("saved:", architecture_table_path)

## 2. Architecture comparison figure

This figure turns the table into the core visual claim.

It replaces the old \(N\)-versus-\(N\) plot with an architecture diagram.

In [ ]:
def draw_box(ax, xy, text, width=0.34, height=0.13, fontsize=11):
    x, y = xy
    box = FancyBboxPatch((x, y), width, height, boxstyle="round,pad=0.025,rounding_size=0.025",
                         linewidth=1.5, facecolor="white", edgecolor="black")
    ax.add_patch(box)
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)

def draw_arrow(ax, x, y_top, y_bottom):
    ax.annotate("", xy=(x, y_bottom), xytext=(x, y_top), arrowprops=dict(arrowstyle="->", linewidth=1.8))

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(0.25, 0.94, "Scale by Devices", ha="center", va="center", fontsize=16, fontweight="bold")
ax.text(0.75, 0.94, "Scale by Modes", ha="center", va="center", fontsize=16, fontweight="bold")

left_x, right_x = 0.08, 0.58
y_positions = [0.76, 0.56, 0.36, 0.16]

for y, label in zip(y_positions, ["N channels", "N sources", "N detectors", "N optical paths"]):
    draw_box(ax, (left_x, y), label)

for y, label in zip(y_positions, ["N channels", "1 resonator", "N frequency modes", "N entangled channels"]):
    draw_box(ax, (right_x, y), label)

for ys in zip(y_positions[:-1], y_positions[1:]):
    draw_arrow(ax, left_x + 0.17, ys[0], ys[1] + 0.13)
    draw_arrow(ax, right_x + 0.17, ys[0], ys[1] + 0.13)

ax.text(0.5, 0.50, "vs", ha="center", va="center", fontsize=18, fontweight="bold")
ax.text(0.5, 0.04, "Frequency multiplexing changes the scaling resource: devices → modes.",
        ha="center", va="center", fontsize=12, fontweight="bold")

fig.tight_layout()
architecture_figure_path = FIGURES_DIR / "00_v2_architecture_comparison.png"
fig.savefig(architecture_figure_path, dpi=200)
plt.show()
print("saved:", architecture_figure_path)

## 3. Resource substitution

The old figure accidentally compared \(N\) sources to \(N\) modes.

This v2 section compares the **hardware resource**:

\[
\text{device scaling sources} = N
\]

\[
\text{mode scaling pump sources} = 1
\]

In [ ]:
channels = np.arange(1, 101)

resources = pd.DataFrame({
    "channels": channels,
    "device_scaled_sources": channels,
    "device_scaled_resonators": channels,
    "device_scaled_optical_paths": channels,
    "mode_scaled_pump_sources": np.ones_like(channels),
    "mode_scaled_resonators": np.ones_like(channels),
    "mode_scaled_frequency_modes": channels,
    "mode_scaled_channels": channels,
})

resources_path = CSV_DIR / "00_v2_resource_substitution.csv"
resources.to_csv(resources_path, index=False)
resources.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(resources["channels"], resources["device_scaled_sources"], label="Scale by devices: sources required", linewidth=2)
ax.plot(resources["channels"], resources["mode_scaled_pump_sources"], label="Scale by modes: pump sources required", linewidth=2)

ax.set_title("Source Count: Replication vs Multiplexing")
ax.set_xlabel("Quantum channels")
ax.set_ylabel("Source count")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

source_count_path = FIGURES_DIR / "00_v2_source_count_substitution.png"
fig.savefig(source_count_path, dpi=200)
plt.show()
print("saved:", source_count_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(resources["channels"], resources["device_scaled_resonators"], label="Scale by devices: resonators/devices required", linewidth=2)
ax.plot(resources["channels"], resources["mode_scaled_resonators"], label="Scale by modes: resonators required", linewidth=2)

ax.set_title("Resonator Count: Replication vs Multiplexing")
ax.set_xlabel("Quantum channels")
ax.set_ylabel("Resonator / device count")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

resonator_count_path = FIGURES_DIR / "00_v2_resonator_count_substitution.png"
fig.savefig(resonator_count_path, dpi=200)
plt.show()
print("saved:", resonator_count_path)

## 4. Frequency comb architecture

A microcomb supplies a ladder of frequency modes:

\[
f_n = f_0 + n\Delta f
\]

This notebook shows the comb architecture only.

Kerr-pair labels such as \((-1,+1)\), \((-2,+2)\), and \((-3,+3)\) belong in Notebook 13.

In [ ]:
mode_indices = np.arange(-10, 11)

mode_labels = []
for n in mode_indices:
    if n == 0:
        mode_labels.append("f₀")
    elif n < 0:
        mode_labels.append(f"f₀{n}Δf")
    else:
        mode_labels.append(f"f₀+{n}Δf")

mode_table = pd.DataFrame({
    "mode_index_n": mode_indices,
    "relative_frequency": mode_indices,
    "frequency_label": mode_labels,
    "role": np.where(mode_indices == 0, "pump", "frequency mode")
})

mode_table_path = CSV_DIR / "00_v2_frequency_comb_modes.csv"
mode_table.to_csv(mode_table_path, index=False)
mode_table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

heights = np.ones_like(mode_indices, dtype=float)
ax.vlines(mode_indices, 0, heights, linewidth=1)
ax.scatter(mode_indices, heights, s=28)

ax.axvline(0, linestyle="--", alpha=0.6)
ax.text(0, 1.12, "pump\nf₀", ha="center", va="bottom")

# Pair links are shown only as geometry, without center text labels.
for idx, n in enumerate(range(1, 6)):
    y = 0.76 - idx * 0.08
    ax.plot([-n, n], [y, y], linewidth=1.2)
    ax.scatter([-n, n], [y, y], s=10)

tick_positions = [-10, -5, -1, 0, 1, 5, 10]
tick_labels = ["f₀−10Δf", "f₀−5Δf", "f₀−Δf", "f₀", "f₀+Δf", "f₀+5Δf", "f₀+10Δf"]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Frequency Comb Architecture")
ax.set_xlabel("Frequency mode")
ax.set_yticks([])
ax.set_ylim(0, 1.25)
ax.set_xlim(mode_indices.min() - 1, mode_indices.max() + 1)
fig.tight_layout()

frequency_comb_path = FIGURES_DIR / "00_v2_frequency_comb.png"
fig.savefig(frequency_comb_path, dpi=200)
plt.show()
print("saved:", frequency_comb_path)

## 5. Summary

Classical scaling increases physical device count.

Microcomb scaling increases frequency-mode count.

This repository explores whether frequency multiplexing changes the scaling architecture of quantum systems.

In [ ]:
summary = {
    "notebook": "00_context_v2",
    "version": VERSION,
    "title": "Frequency Multiplexing Enables Quantum Scale",
    "seminar": "Integrated Microcombs for Quantum Applications",
    "speaker": "Xu Yi, University of Virginia",
    "repo_question": "Which resource scales quantum systems: more devices or more modes?",
    "architectural_claim": "Frequency multiplexing changes the scaling resource from replicated devices to frequency-indexed modes.",
    "outputs": [
        "figures/00_v2_architecture_comparison.png",
        "figures/00_v2_source_count_substitution.png",
        "figures/00_v2_resonator_count_substitution.png",
        "figures/00_v2_frequency_comb.png",
        "results/csv/00_v2_architecture_table.csv",
        "results/csv/00_v2_resource_substitution.csv",
        "results/csv/00_v2_frequency_comb_modes.csv",
        "results/json/00_v2_context_summary.json"
    ]
}

summary_path = JSON_DIR / "00_v2_context_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

In [ ]:
outputs = [
    FIGURES_DIR / "00_v2_architecture_comparison.png",
    FIGURES_DIR / "00_v2_source_count_substitution.png",
    FIGURES_DIR / "00_v2_resonator_count_substitution.png",
    FIGURES_DIR / "00_v2_frequency_comb.png",
    CSV_DIR / "00_v2_architecture_table.csv",
    CSV_DIR / "00_v2_resource_substitution.csv",
    CSV_DIR / "00_v2_frequency_comb_modes.csv",
    JSON_DIR / "00_v2_context_summary.json",
]

for output in outputs:
    print("exists:", output.exists(), "→", output.relative_to(ROOT) if output.exists() else output)

## Takeaway

The notebook's central result is architectural:

\[
\text{Scale by devices:}\quad N\ \text{channels} \rightarrow N\ \text{source paths}
\]

\[
\text{Scale by modes:}\quad 1\ \text{resonator} \rightarrow N\ \text{frequency modes} \rightarrow N\ \text{channels}
\]

This motivates the next notebooks:

- **07:** frequency comb architecture
- **13:** Kerr pair generation
- **23:** multipartite entanglement networks
- **37:** scaling by modes vs devices